# Olist — Treinamento do Modelo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import joblib
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

PASTA = 'C:/Users/eymar/Downloads/olist/archive/'

X_train = pd.read_csv(PASTA + 'X_train.csv')
X_test  = pd.read_csv(PASTA + 'X_test.csv')
y_train = pd.read_csv(PASTA + 'y_train.csv').squeeze()
y_test  = pd.read_csv(PASTA + 'y_test.csv').squeeze()

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  |  y_test:  {y_test.shape}')
print(f'\nFeatures: {X_train.shape[1]}')
print(f'Target  : dias_entrega  —  media={y_train.mean():.1f}  desvio={y_train.std():.1f}')

# Lista para acumular resultados de todos os modelos
resultados = []

# Funcao de avaliacao (usada em todos os modelos)
def avaliar(nome, modelo, X_tr, y_tr, X_te, y_te):
    y_pred_tr = modelo.predict(X_tr)
    y_pred_te = modelo.predict(X_te)
    res = {
        'Modelo'     : nome,
        'MAE treino' : round(mean_absolute_error(y_tr, y_pred_tr), 3),
        'MAE teste'  : round(mean_absolute_error(y_te, y_pred_te), 3),
        'RMSE treino': round(np.sqrt(mean_squared_error(y_tr, y_pred_tr)), 3),
        'RMSE teste' : round(np.sqrt(mean_squared_error(y_te, y_pred_te)), 3),
        'R2 treino'  : round(r2_score(y_tr, y_pred_tr), 4),
        'R2 teste'   : round(r2_score(y_te, y_pred_te), 4),
    }
    resultados.append(res)
    return res

# Funcao de graficos de diagnostico
def plot_diagnostico(nome, y_te, y_pred_te):
    residuos = y_te.values - y_pred_te
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'{nome} — Diagnostico no Conjunto de Teste', fontweight='bold')

    # Previsto vs Real
    lim = [0, max(y_te.max(), y_pred_te.max()) + 2]
    axes[0].scatter(y_te, y_pred_te, alpha=0.15, s=8, color='steelblue')
    axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Previsao perfeita')
    axes[0].set_xlim(lim); axes[0].set_ylim(lim)
    axes[0].set_xlabel('Real (dias)'); axes[0].set_ylabel('Previsto (dias)')
    axes[0].set_title('Previsto vs Real'); axes[0].legend()

    # Distribuicao dos erros
    axes[1].hist(residuos, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Erro zero')
    axes[1].set_xlabel('Erro (dias)'); axes[1].set_ylabel('Frequencia')
    axes[1].set_title(f'Distribuicao dos Erros  |  media={residuos.mean():.2f}  desvio={residuos.std():.2f}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 1. Ridge

In [ ]:
print('=== Modelo 1: Ridge ===\n')
t0 = time.time()

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

print(f'Tempo de treino: {time.time() - t0:.2f}s')

y_pred_ridge = ridge.predict(X_test)
res = avaliar('Ridge', ridge, X_train, y_train, X_test, y_test)

print(f'\nResultados no conjunto de teste:')
print(f'  MAE  = {res["MAE teste"]:.3f} dias')
print(f'  RMSE = {res["RMSE teste"]:.3f} dias')
print(f'  R2   = {res["R2 teste"]:.4f}')
print(f'\nO modelo erra em media {res["MAE teste"]:.1f} dia(s) por pedido.')
print(f'Overfitting (delta MAE treino->teste): {res["MAE teste"] - res["MAE treino"]:+.3f}')

plot_diagnostico('Ridge', y_test, y_pred_ridge)

## 2. Random Forest

In [ ]:
print('=== Modelo 2: Random Forest ===\n')
print('Treinando 100 arvores — pode demorar alguns segundos...')
t0 = time.time()

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

print(f'Tempo de treino: {time.time() - t0:.1f}s')

y_pred_rf = rf.predict(X_test)
res = avaliar('Random Forest', rf, X_train, y_train, X_test, y_test)

print(f'\nResultados no conjunto de teste:')
print(f'  MAE  = {res["MAE teste"]:.3f} dias')
print(f'  RMSE = {res["RMSE teste"]:.3f} dias')
print(f'  R2   = {res["R2 teste"]:.4f}')
print(f'\nO modelo erra em media {res["MAE teste"]:.1f} dia(s) por pedido.')
print(f'Overfitting (delta MAE treino->teste): {res["MAE teste"] - res["MAE treino"]:+.3f}')

plot_diagnostico('Random Forest', y_test, y_pred_rf)

## 3. XGBoost

In [ ]:
print('=== Modelo 3: XGBoost ===\n')
print('Treinando 300 arvores em sequencia...')
t0 = time.time()

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(X_train, y_train)

print(f'Tempo de treino: {time.time() - t0:.1f}s')

y_pred_xgb = xgb.predict(X_test)
res = avaliar('XGBoost', xgb, X_train, y_train, X_test, y_test)

print(f'\nResultados no conjunto de teste:')
print(f'  MAE  = {res["MAE teste"]:.3f} dias')
print(f'  RMSE = {res["RMSE teste"]:.3f} dias')
print(f'  R2   = {res["R2 teste"]:.4f}')
print(f'\nO modelo erra em media {res["MAE teste"]:.1f} dia(s) por pedido.')
print(f'Overfitting (delta MAE treino->teste): {res["MAE teste"] - res["MAE treino"]:+.3f}')

plot_diagnostico('XGBoost', y_test, y_pred_xgb)

## 4. Comparacao dos Modelos

In [ ]:
df_res = pd.DataFrame(resultados).set_index('Modelo')

print('Comparacao completa (treino e teste):\n')
print(df_res.to_string())

# Graficos de comparacao
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparacao dos Modelos — Conjunto de Teste', fontweight='bold', fontsize=13)

cores   = ['#5b9bd5', '#70ad47', '#ed7d31']
modelos = df_res.index.tolist()

for ax, metrica, titulo, melhor in [
    (axes[0], 'MAE teste',  'MAE — menor e melhor',  'min'),
    (axes[1], 'RMSE teste', 'RMSE — menor e melhor', 'min'),
    (axes[2], 'R2 teste',   'R2 — maior e melhor',   'max'),
]:
    valores = df_res[metrica].values
    bars = ax.bar(modelos, valores, color=cores, alpha=0.85, edgecolor='white', width=0.5)
    idx_melhor = int(np.argmin(valores) if melhor == 'min' else np.argmax(valores))
    bars[idx_melhor].set_edgecolor('gold')
    bars[idx_melhor].set_linewidth(3)
    for bar, v in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2,
                v + max(valores) * 0.01,
                f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel(metrica)
    ax.tick_params(axis='x', labelrotation=10)

plt.tight_layout()
plt.show()

# Identificar melhor
melhor_mae  = df_res['MAE teste'].idxmin()
print(f'\nMelhor modelo (menor MAE no teste): {melhor_mae}')
print(f'  MAE  = {df_res.loc[melhor_mae, "MAE teste"]:.3f} dias')
print(f'  RMSE = {df_res.loc[melhor_mae, "RMSE teste"]:.3f} dias')
print(f'  R2   = {df_res.loc[melhor_mae, "R2 teste"]:.4f}')

# Analise de overfitting
print('\nAnalise de overfitting (treino vs teste):')
for nome in df_res.index:
    delta = df_res.loc[nome, 'MAE teste'] - df_res.loc[nome, 'MAE treino']
    status = 'OK' if delta < 1.5 else 'ATENCAO — possivel overfitting'
    print(f'  {nome:<20s}  delta MAE = {delta:+.3f}  {status}')

## 5. Importancia das Features

In [ ]:
melhor_nome  = df_res['MAE teste'].idxmin()
modelos_dict = {'Ridge': ridge, 'Random Forest': rf, 'XGBoost': xgb}
melhor_mod   = modelos_dict[melhor_nome]

if melhor_nome == 'Ridge':
    importances = pd.Series(np.abs(melhor_mod.coef_), index=X_train.columns)
    label_y = 'Coeficiente (valor absoluto)'
else:
    importances = pd.Series(melhor_mod.feature_importances_, index=X_train.columns)
    label_y = 'Importancia'

# Top 20
top20 = importances.sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['tomato' if v >= top20.values[-3:].min() else 'steelblue' for v in top20.values]
bars = ax.barh(top20.index, top20.values, color=colors, alpha=0.85)
for bar, v in zip(bars, top20.values):
    ax.text(v + max(top20.values) * 0.005,
            bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=8)
ax.set_title(f'Top 20 Features — {melhor_nome}\n(vermelho = top 3)', fontweight='bold')
ax.set_xlabel(label_y)
plt.tight_layout()
plt.show()

# Tabela completa
print(f'Ranking completo de importancia — {melhor_nome}:')
print(f'\n{"Feature":<35s}  {label_y}')
print('-' * 60)
for feat, val in importances.sort_values(ascending=False).items():
    barra = '|' * int(val / importances.max() * 25)
    print(f'{feat:<35s}  {val:.4f}  {barra}')

## 6. Analise de Erros

In [ ]:
preds_dict = {'Ridge': y_pred_ridge, 'Random Forest': y_pred_rf, 'XGBoost': y_pred_xgb}
y_pred_melhor = preds_dict[melhor_nome]
residuos = y_test.values - y_pred_melhor

df_err = pd.DataFrame({
    'real'     : y_test.values,
    'previsto' : y_pred_melhor,
    'erro'     : residuos,
    'erro_abs' : np.abs(residuos)
})
df_err['faixa'] = pd.cut(
    df_err['real'],
    bins=[0, 7, 14, 21, 30, 100],
    labels=['1-7d', '8-14d', '15-21d', '22-30d', '30d+']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Analise de Erros — {melhor_nome}', fontweight='bold')

# MAE por faixa de entrega
mae_faixa = df_err.groupby('faixa', observed=True)['erro_abs'].mean()
bars = axes[0].bar(mae_faixa.index, mae_faixa.values, color='steelblue', alpha=0.85)
for bar, v in zip(bars, mae_faixa.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.05,
                 f'{v:.1f}d', ha='center', fontsize=10)
axes[0].set_title('MAE por faixa de entrega real')
axes[0].set_xlabel('Faixa de dias reais')
axes[0].set_ylabel('Erro medio absoluto (dias)')

# Scatter previsto vs real (colorido pelo erro absoluto)
sc = axes[1].scatter(df_err['real'], df_err['previsto'],
                     c=df_err['erro_abs'], cmap='YlOrRd',
                     alpha=0.3, s=8, vmax=df_err['erro_abs'].quantile(0.95))
lim = [0, df_err['real'].max() + 2]
axes[1].plot(lim, lim, 'b--', linewidth=1.5, label='Ideal')
axes[1].set_xlabel('Real (dias)'); axes[1].set_ylabel('Previsto (dias)')
axes[1].set_title('Previsto vs Real — cor indica magnitude do erro')
axes[1].legend()
plt.colorbar(sc, ax=axes[1], label='Erro absoluto (dias)')

plt.tight_layout()
plt.show()

# Estatisticas
print(f'Estatisticas dos erros — {melhor_nome} (teste):')
print(f'  Media dos erros:           {residuos.mean():.3f} dias')
print(f'  Desvio dos erros:          {residuos.std():.3f} dias')
print(f'  Mediana do erro absoluto:  {np.median(np.abs(residuos)):.3f} dias')
print()
print(f'  Pedidos com erro < 2 dias: {(np.abs(residuos) < 2).mean()*100:.1f}%')
print(f'  Pedidos com erro < 3 dias: {(np.abs(residuos) < 3).mean()*100:.1f}%')
print(f'  Pedidos com erro < 5 dias: {(np.abs(residuos) < 5).mean()*100:.1f}%')
print(f'  Pedidos com erro < 7 dias: {(np.abs(residuos) < 7).mean()*100:.1f}%')

## 7. Conclusao e Proximo Passo

In [ ]:
# Salvar o melhor modelo
joblib.dump(melhor_mod, PASTA + 'modelo_final.pkl')
print(f'Modelo salvo: {PASTA}modelo_final.pkl')

# Resumo final
print()
print('=' * 65)
print('  CONCLUSAO DO TREINAMENTO')
print('=' * 65)
print(f'\nDataset de treino:  {X_train.shape[0]:,} pedidos')
print(f'Dataset de teste:   {X_test.shape[0]:,} pedidos')
print(f'Features:           {X_train.shape[1]}')
print()
print('Desempenho no conjunto de TESTE:')
print(df_res[['MAE teste', 'RMSE teste', 'R2 teste']].round(3).to_string())

print()
print(f'Melhor modelo: {melhor_nome}')
print(f'  Erra em media {df_res.loc[melhor_nome, "MAE teste"]:.1f} dia(s) por pedido')
print(f'  Explica {df_res.loc[melhor_nome, "R2 teste"]*100:.1f}% da variacao do tempo de entrega')

print()
print('Analise de overfitting (delta MAE treino -> teste):')
for nome in df_res.index:
    delta  = df_res.loc[nome, 'MAE teste'] - df_res.loc[nome, 'MAE treino']
    status = 'OK' if delta < 1.5 else 'ATENCAO'
    print(f'  {nome:<20s}  {delta:+.3f} dias  [{status}]')

print()
print('Proximo passo: ajuste de hiperparametros com GridSearchCV')
print('=' * 65)